# Hybrid QA Retrieval Baseline v2.1

This notebook builds a stronger baseline for the Vietnamese culture QA agent.

Main idea:

```text
raw VQA dataset
-> compact qa_core_chunk, anchored by keyword/topic terms
-> topic_card per cultural topic
-> E5 embeddings + Chroma
-> vector search + lexical rerank
-> zip Chroma DB for local use
```

Why this differs from the previous clean QA notebook:

- Keep a strong `retrieval_anchor` from `keyword`, because it helps entity matching.
- Keep `canonical_topic` separately, so noisy keywords do not become factual truth.
- Use compact embedding text instead of stuffing every long field into each chunk.
- Add `topic_card` for general questions and recommendations.
- Rerank results with lexical overlap so entity words like `thảm cói` matter.

In [ ]:
!pip install -q langchain-core langchain-chroma langchain-huggingface sentence-transformers chromadb

In [ ]:
# =============================================================================
# 1. Imports and configuration
# =============================================================================

from __future__ import annotations

import json
import os
import random
import re
import shutil
import unicodedata
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import torch
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


# Set manually if auto-discovery cannot find your dataset.
DATASET_PATH = None

PERSIST_DIR = Path('/kaggle/working/chroma_db_qa_hybrid')
ZIP_PATH = Path('/kaggle/working/chroma_db_qa_hybrid.zip')

COLLECTION_NAME = 'qa_hybrid_chunks'
EMBED_MODEL = 'intfloat/multilingual-e5-base'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Keep the index focused. Set to None only if you really want every sample.
MAX_SAMPLES_PER_ANCHOR = 8
QA_PER_SAMPLE = 4

# Use a small number like 500 for a smoke test. Use None for full selected corpus.
RAW_SAMPLE_LIMIT = None

INDEX_BATCH_SIZE = 512
EMBED_BATCH_SIZE = 64
CLEAR_EXISTING_INDEX = True
SEED = 42

random.seed(SEED)

print('Device:', DEVICE)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# =============================================================================
# 2. Dataset and token helpers
# =============================================================================

def load_hf_token_from_kaggle_secret() -> None:
    """Load HF_TOKEN from Kaggle Secrets if available."""

    if os.environ.get('HF_TOKEN'):
        return

    try:
        from kaggle_secrets import UserSecretsClient
    except Exception:
        return

    secret_client = UserSecretsClient()

    for secret_name in ('HF_TOKEN', 'HUGGINGFACEHUB_API_TOKEN'):
        try:
            token = secret_client.get_secret(secret_name)
        except Exception:
            token = None

        if token:
            os.environ['HF_TOKEN'] = token
            return


def find_dataset_path() -> Path:
    """Find vietnamese_vqa_dataset.json under /kaggle/input."""

    if DATASET_PATH:
        return Path(DATASET_PATH)

    candidates = list(Path('/kaggle/input').rglob('vietnamese_vqa_dataset.json'))
    if not candidates:
        raise FileNotFoundError('Set DATASET_PATH manually; dataset was not found.')

    return candidates[0]


def load_dataset(dataset_path: str | Path) -> list[dict[str, Any]]:
    """Load raw JSON records."""

    with Path(dataset_path).open('r', encoding='utf-8') as dataset_file:
        return json.load(dataset_file)


load_hf_token_from_kaggle_secret()
os.environ.setdefault('HF_HOME', '/kaggle/working/hf_cache')
os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS_WARNING', '1')

dataset_path = find_dataset_path()
dataset_records = load_dataset(dataset_path)

if RAW_SAMPLE_LIMIT is not None:
    dataset_records = dataset_records[:RAW_SAMPLE_LIMIT]

print('Dataset path:', dataset_path)
print('Raw records:', len(dataset_records))
print('HF token loaded:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
# =============================================================================
# 3. Text helpers and topic logic
# =============================================================================

GENERIC_TOPIC_PREFIXES = (
    'mot', 'nhung', 'hinh anh', 'canh', 'qua trinh', 'hoat dong', 'nguoi',
)

STOPWORDS = {
    'la', 'gi', 'cua', 've', 'va', 'trong', 'nay', 'do', 'mot', 'nhung',
    'cac', 'co', 'y', 'nghia', 'van', 'hoa', 'tai', 'sao', 'the', 'nao',
    'hinh', 'anh', 'viet', 'nam', 'vietnam', 'vietnamese', 'traditional',
}


@dataclass(frozen=True)
class TopicInfo:
    retrieval_anchor: str
    normalized_anchor: str
    canonical_topic: str
    topic_source: str
    aliases: list[str]


@dataclass(frozen=True)
class HybridChunk:
    page_content: str
    metadata: dict[str, Any]


def safe_text(value: Any) -> str:
    """Convert optional strings/lists into display text."""

    if value is None:
        return ''

    if isinstance(value, list):
        return ', '.join(str(item).strip() for item in value if item)

    return str(value).strip()


def normalize_text(value: Any) -> str:
    """Normalize text for matching and dedup only."""

    text = safe_text(value).lower()
    if not text:
        return ''

    text = unicodedata.normalize('NFC', text)
    text = ''.join(
        char
        for char in unicodedata.normalize('NFD', text)
        if unicodedata.category(char) != 'Mn'
    )
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


def tokenize_for_rerank(value: Any) -> set[str]:
    """Tokenize Vietnamese-ish text for simple lexical rerank."""

    tokens = set(normalize_text(value).split())
    return {token for token in tokens if token not in STOPWORDS and len(token) > 1}


def is_generic_topic(topic: str) -> bool:
    """Reject topics that are too visual/generic for retrieval anchoring."""

    normalized_topic = normalize_text(topic)
    if not normalized_topic:
        return True

    if len(normalized_topic.split()) > 12:
        return True

    return any(normalized_topic.startswith(prefix) for prefix in GENERIC_TOPIC_PREFIXES)


def clean_display_topic(value: Any) -> str:
    """Clean punctuation while preserving Vietnamese accents."""

    return safe_text(value).strip(' .,:;!?')


def get_primary_object_topic(sample: dict[str, Any]) -> tuple[str, str] | None:
    """Prefer cultural objects over generic visual answers."""

    cultural_context = sample.get('cultural_context', {}) or {}
    primary_objects = cultural_context.get('primary_cultural_objects', []) or []

    for object_name in primary_objects:
        candidate = clean_display_topic(object_name)
        if candidate and not is_generic_topic(candidate):
            return candidate, 'primary_cultural_object'

    return None


def get_identification_topic(sample: dict[str, Any]) -> tuple[str, str] | None:
    """Use identification answer only when it is specific enough."""

    for question_record in sample.get('questions', []) or []:
        question_type = normalize_text(question_record.get('question_type'))
        if 'identification' not in question_type:
            continue

        candidate = clean_display_topic(question_record.get('answer'))
        if candidate and not is_generic_topic(candidate):
            return candidate, 'identification_answer'

    return None


def build_aliases(sample: dict[str, Any], retrieval_anchor: str, canonical_topic: str) -> list[str]:
    """Collect terms that should help vector retrieval and reranking."""

    cultural_context = sample.get('cultural_context', {}) or {}
    primary_objects = cultural_context.get('primary_cultural_objects', []) or []

    raw_aliases = [
        retrieval_anchor,
        canonical_topic,
        sample.get('keyword'),
        sample.get('category'),
        cultural_context.get('cultural_category'),
        *primary_objects,
    ]

    aliases = []
    seen = set()

    for raw_alias in raw_aliases:
        alias = safe_text(raw_alias)
        normalized_alias = normalize_text(alias)

        if not alias or not normalized_alias or normalized_alias in seen:
            continue

        aliases.append(alias)
        seen.add(normalized_alias)

    return aliases


def resolve_topic_info(sample: dict[str, Any]) -> TopicInfo:
    """Keep keyword as retrieval anchor, but keep factual topic separately."""

    retrieval_anchor = clean_display_topic(sample.get('keyword'))
    normalized_anchor = normalize_text(retrieval_anchor)

    topic_pair = get_primary_object_topic(sample) or get_identification_topic(sample)
    if topic_pair:
        canonical_topic, topic_source = topic_pair
    else:
        canonical_topic, topic_source = retrieval_anchor, 'keyword_fallback'

    aliases = build_aliases(sample, retrieval_anchor, canonical_topic)

    return TopicInfo(
        retrieval_anchor=retrieval_anchor,
        normalized_anchor=normalized_anchor,
        canonical_topic=canonical_topic,
        topic_source=topic_source,
        aliases=aliases,
    )

In [ ]:
# =============================================================================
# 4. Sampling and QA selection
# =============================================================================

PREFERRED_QUESTION_TYPES = [
    'identification',
    'description',
    'cultural',
    'analysis',
    'comparison',
    'historical',
]


def group_records_by_anchor(records: list[dict[str, Any]]) -> dict[tuple[str, str], list[dict[str, Any]]]:
    """Group samples by category + keyword anchor."""

    groups = defaultdict(list)

    for sample in records:
        topic_info = resolve_topic_info(sample)
        group_key = (safe_text(sample.get('category')), topic_info.normalized_anchor)
        groups[group_key].append(sample)

    return groups


def select_records(records: list[dict[str, Any]], max_per_anchor: int | None) -> list[dict[str, Any]]:
    """Keep a focused but broad corpus instead of indexing every near-duplicate."""

    if max_per_anchor is None:
        return records

    selected_records = []
    grouped_records = group_records_by_anchor(records)

    for _, group in grouped_records.items():
        random.shuffle(group)
        selected_records.extend(group[:max_per_anchor])

    return selected_records


def pick_diverse_questions(questions: list[dict[str, Any]], max_questions: int) -> list[dict[str, Any]]:
    """Pick diverse question types from one sample."""

    selected = []
    seen_questions = set()

    for target_type in PREFERRED_QUESTION_TYPES:
        for question_record in questions:
            question_type = safe_text(question_record.get('question_type'))
            normalized_question = normalize_text(question_record.get('question'))

            if target_type not in normalize_text(question_type):
                continue

            if not normalized_question or normalized_question in seen_questions:
                continue

            selected.append(question_record)
            seen_questions.add(normalized_question)
            break

        if len(selected) >= max_questions:
            return selected

    for question_record in questions:
        if len(selected) >= max_questions:
            break

        normalized_question = normalize_text(question_record.get('question'))
        if not normalized_question or normalized_question in seen_questions:
            continue

        selected.append(question_record)
        seen_questions.add(normalized_question)

    return selected

In [ ]:
# =============================================================================
# 5. Build compact QA chunks and topic cards
# =============================================================================

def build_search_terms(topic_info: TopicInfo) -> str:
    """Create compact search terms near the top of every embedded chunk."""

    return '; '.join(topic_info.aliases)


def build_qa_core_chunk(sample: dict[str, Any], question_record: dict[str, Any]) -> HybridChunk:
    """Build the main retrieval unit: compact QA content with strong anchors."""

    topic_info = resolve_topic_info(sample)

    question_text = safe_text(question_record.get('question'))
    answer_text = safe_text(question_record.get('answer'))
    explanation_text = safe_text(question_record.get('detailed_explanation'))
    significance_text = safe_text(question_record.get('cultural_significance'))

    page_content = f"""
Chunk Type: qa_core_chunk

Search Terms:
{build_search_terms(topic_info)}

Topic Anchor:
{topic_info.retrieval_anchor}

Canonical Topic:
{topic_info.canonical_topic}

Category:
{safe_text(sample.get('category'))}

Question Type:
{safe_text(question_record.get('question_type'))}

Representative Question:
{question_text}

Answer:
{answer_text}

Detailed Explanation:
{explanation_text}

Cultural Significance:
{significance_text}
""".strip()

    metadata = {
        'chunk_type': 'qa_core_chunk',
        'category': safe_text(sample.get('category')),
        'retrieval_anchor': topic_info.retrieval_anchor,
        'normalized_anchor': topic_info.normalized_anchor,
        'canonical_topic': topic_info.canonical_topic,
        'topic_source': topic_info.topic_source,
        'aliases': ' | '.join(topic_info.aliases),
        'keyword': safe_text(sample.get('keyword')),
        'normalized_keyword': normalize_text(sample.get('keyword')),
        'question': question_text,
        'normalized_question': normalize_text(question_text),
        'question_type': safe_text(question_record.get('question_type')),
        'difficulty': safe_text(question_record.get('difficulty')),
        'cognitive_level': safe_text(question_record.get('cognitive_level')),
        'image_id': safe_text(sample.get('image_id')),
        'image_path': safe_text(sample.get('image_path')),
    }

    return HybridChunk(page_content=page_content, metadata=metadata)


def build_topic_card(group_key: tuple[str, str], samples: list[dict[str, Any]]) -> HybridChunk:
    """Build one aggregated topic card per category + anchor."""

    first_sample = samples[0]
    topic_info = resolve_topic_info(first_sample)
    category, normalized_anchor = group_key

    common_questions = []
    answer_snippets = []
    cultural_snippets = []

    seen_questions = set()

    for sample in samples[:6]:
        for question_record in pick_diverse_questions(sample.get('questions', []) or [], max_questions=3):
            normalized_question = normalize_text(question_record.get('question'))
            if normalized_question and normalized_question not in seen_questions:
                common_questions.append(safe_text(question_record.get('question')))
                seen_questions.add(normalized_question)

            answer = safe_text(question_record.get('answer'))
            if answer:
                answer_snippets.append(answer)

            significance = safe_text(question_record.get('cultural_significance'))
            if significance:
                cultural_snippets.append(significance)

    page_content = f"""
Chunk Type: topic_card

Search Terms:
{build_search_terms(topic_info)}

Topic Anchor:
{topic_info.retrieval_anchor}

Canonical Topic:
{topic_info.canonical_topic}

Category:
{category}

Common Questions:
{' | '.join(common_questions[:8])}

Answer Snippets:
{' '.join(dict.fromkeys(answer_snippets[:8]))}

Cultural Themes:
{' '.join(dict.fromkeys(cultural_snippets[:6]))}
""".strip()

    metadata = {
        'chunk_type': 'topic_card',
        'category': category,
        'retrieval_anchor': topic_info.retrieval_anchor,
        'normalized_anchor': normalized_anchor,
        'canonical_topic': topic_info.canonical_topic,
        'topic_source': topic_info.topic_source,
        'aliases': ' | '.join(topic_info.aliases),
        'keyword': safe_text(first_sample.get('keyword')),
        'normalized_keyword': normalize_text(first_sample.get('keyword')),
        'question': '',
        'normalized_question': '',
        'question_type': 'topic_overview',
        'difficulty': '',
        'cognitive_level': '',
        'image_id': safe_text(first_sample.get('image_id')),
        'image_path': safe_text(first_sample.get('image_path')),
    }

    return HybridChunk(page_content=page_content, metadata=metadata)


def build_hybrid_chunks(records: list[dict[str, Any]]) -> list[HybridChunk]:
    """Build compact QA chunks plus topic cards."""

    selected_records = select_records(records, MAX_SAMPLES_PER_ANCHOR)
    grouped_records = group_records_by_anchor(selected_records)

    chunks = []
    seen_qa_keys = set()

    for group_key, samples in grouped_records.items():
        chunks.append(build_topic_card(group_key, samples))

        for sample in samples:
            topic_info = resolve_topic_info(sample)
            selected_questions = pick_diverse_questions(sample.get('questions', []) or [], QA_PER_SAMPLE)

            for question_record in selected_questions:
                qa_key = (
                    safe_text(sample.get('category')),
                    topic_info.normalized_anchor,
                    safe_text(question_record.get('question_type')),
                    normalize_text(question_record.get('question')),
                )

                if qa_key in seen_qa_keys:
                    continue

                chunks.append(build_qa_core_chunk(sample, question_record))
                seen_qa_keys.add(qa_key)

    return chunks


def summarize_chunks(chunks: list[HybridChunk]) -> None:
    """Print compact summary before embedding."""

    print('Total chunks:', len(chunks))
    print('Chunk types:', dict(Counter(chunk.metadata['chunk_type'] for chunk in chunks)))
    print('Categories:', Counter(chunk.metadata['category'] for chunk in chunks).most_common())
    print('Topic sources:', dict(Counter(chunk.metadata['topic_source'] for chunk in chunks)))


def preview_chunks(chunks: list[HybridChunk], limit: int = 3) -> None:
    """Preview chunks before indexing."""

    for index, chunk in enumerate(chunks[:limit], start=1):
        metadata = chunk.metadata
        print('=' * 80)
        print('CHUNK', index)
        print('chunk_type:', metadata['chunk_type'])
        print('retrieval_anchor:', metadata['retrieval_anchor'])
        print('canonical_topic:', metadata['canonical_topic'])
        print('keyword:', metadata['keyword'])
        print('question:', metadata['question'])
        print(chunk.page_content[:900])


hybrid_chunks = build_hybrid_chunks(dataset_records)
summarize_chunks(hybrid_chunks)
preview_chunks(hybrid_chunks, limit=5)

documents = [Document(page_content=chunk.page_content, metadata=chunk.metadata) for chunk in hybrid_chunks]
print('Documents ready:', len(documents))

In [ ]:
# =============================================================================
# 6. Build Chroma index
# =============================================================================

class E5Embeddings:
    """Add E5 prefixes for passages and queries."""

    def __init__(self, model_name: str, device: str) -> None:
        self.embedding_model = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': device},
            encode_kwargs={
                'normalize_embeddings': True,
                'batch_size': EMBED_BATCH_SIZE,
            },
        )

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        passages = [f'passage: {text}' for text in texts]
        return self.embedding_model.embed_documents(passages)

    def embed_query(self, text: str) -> list[float]:
        return self.embedding_model.embed_query(f'query: {text}')


def batch_documents(items: list[Document], batch_size: int):
    """Yield batches so indexing progress is visible."""

    for start_index in range(0, len(items), batch_size):
        yield start_index, items[start_index:start_index + batch_size]


if CLEAR_EXISTING_INDEX and PERSIST_DIR.exists():
    shutil.rmtree(PERSIST_DIR)

PERSIST_DIR.mkdir(parents=True, exist_ok=True)

embeddings = E5Embeddings(model_name=EMBED_MODEL, device=DEVICE)
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIR),
)

for start_index, document_batch in batch_documents(documents, INDEX_BATCH_SIZE):
    vectorstore.add_documents(document_batch)
    indexed_count = min(start_index + len(document_batch), len(documents))
    print(f'Indexed {indexed_count}/{len(documents)} documents')

print('Chroma index saved to:', PERSIST_DIR)

In [ ]:
# =============================================================================
# 7. Vector search + lexical rerank
# =============================================================================

def lexical_overlap_score(query: str, metadata: dict[str, Any]) -> float:
    """Reward exact/lexical overlap with anchor, topic, aliases, and question."""

    query_tokens = tokenize_for_rerank(query)
    anchor_tokens = tokenize_for_rerank(metadata.get('retrieval_anchor', ''))
    topic_tokens = tokenize_for_rerank(metadata.get('canonical_topic', ''))
    alias_tokens = tokenize_for_rerank(metadata.get('aliases', ''))
    question_tokens = tokenize_for_rerank(metadata.get('question', ''))

    if not query_tokens:
        return 0.0

    score = 0.0
    score += 3.0 * len(query_tokens & anchor_tokens)
    score += 2.0 * len(query_tokens & topic_tokens)
    score += 1.5 * len(query_tokens & alias_tokens)
    score += 1.0 * len(query_tokens & question_tokens)

    return score


def retrieve_with_rerank(query: str, top_k: int = 5, fetch_k: int = 40):
    """Fetch many vector results, then rerank by lexical entity overlap."""

    raw_results = vectorstore.similarity_search_with_score(query, k=fetch_k)
    reranked_results = []

    for document, vector_distance in raw_results:
        metadata = document.metadata
        lexical_score = lexical_overlap_score(query, metadata)

        # Chroma distance: lower is better. Lexical score: higher is better.
        final_score = float(vector_distance) - (0.04 * lexical_score)

        reranked_results.append({
            'document': document,
            'vector_distance': float(vector_distance),
            'lexical_score': lexical_score,
            'final_score': final_score,
        })

    reranked_results.sort(key=lambda item: item['final_score'])
    return reranked_results[:top_k]


def preview_search(query: str, top_k: int = 5, fetch_k: int = 40) -> None:
    """Print enough metadata to judge retrieval quality."""

    results = retrieve_with_rerank(query, top_k=top_k, fetch_k=fetch_k)

    print('=' * 80)
    print('QUERY:', query)

    for rank, result in enumerate(results, start=1):
        document = result['document']
        metadata = document.metadata

        print('=' * 80)
        print(f"RANK {rank} | final={result['final_score']:.6f} | vector={result['vector_distance']:.6f} | lexical={result['lexical_score']:.2f}")
        print('chunk_type:', metadata.get('chunk_type'))
        print('retrieval_anchor:', metadata.get('retrieval_anchor'))
        print('canonical_topic:', metadata.get('canonical_topic'))
        print('keyword:', metadata.get('keyword'))
        print('category:', metadata.get('category'))
        print('question_type:', metadata.get('question_type'))
        print('question:', metadata.get('question'))
        print(document.page_content[:850])


TEST_QUERIES = [
    'Ý nghĩa văn hóa của thảm cói là gì?',
    'Xe máy là gì?',
    'So sánh bánh chưng và bánh tét',
    'Nguồn gốc của bánh chưng là gì?',
    'Đàn bầu là gì?',
    'Múa rối nước có ý nghĩa gì?',
]

for query in TEST_QUERIES:
    preview_search(query, top_k=5, fetch_k=50)

In [ ]:
# =============================================================================
# 8. Zip Chroma DB for local download
# =============================================================================

def zip_directory(source_dir: Path, output_zip: Path) -> None:
    """Zip the Chroma directory while preserving the folder name."""

    if output_zip.exists():
        output_zip.unlink()

    with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zip_file:
        for file_path in source_dir.rglob('*'):
            if file_path.is_file():
                zip_file.write(file_path, file_path.relative_to(source_dir.parent))


zip_directory(PERSIST_DIR, ZIP_PATH)
print('Created:', ZIP_PATH)
print('Download and extract to D:/Ds107/chroma_db_qa_hybrid')